# Measuring the Gap Between Mental-Health Need and Treatment Capacity in U.S. Counties, 2018–2023

**Author:** Khushi Desai  
**Course:** Data and the State (DATA 20195 / 30195)

---

## Research question

Across U.S. counties, **where does mental-health need outpace the local capacity of the behavioral-health system to treat it, and how has that gap moved over recent years?**

To answer this, the notebook builds a **need–capacity mismatch index** at the county level and tracks it longitudinally (2018–2023). *Need* is measured from population mental-health prevalence; *capacity* is measured from the "reach" of local treatment infrastructure — psychiatric facilities, inpatient psychiatric beds, treatment caseloads, and the number of registered psychiatrists. High mismatch marks places where the state's behavioral-health system is least able to meet the burden it faces.

## Background and framing

Mental-health treatment in the United States is delivered through an uneven patchwork of public and private infrastructure, and the **state's capacity to reach the people who need care varies enormously across geography**. A county can have high measured mental-health burden and almost no inpatient psychiatric beds or psychiatrists within reach; another can have ample infrastructure relative to its need. A simple count of treatment facilities — the measure used in the earlier draft of this project — hides this, because it treats a small outpatient office and a 200-bed psychiatric hospital as equivalent.

This notebook therefore reframes *capacity* as **reach**: not how many facilities exist, but how much treatment a place can actually deliver — beds, wards, caseload, and clinicians. Setting that against modeled mental-health prevalence yields a mismatch index that speaks directly to questions of **state capacity and distributive equity** in behavioral health.

## Data sources

| Source | Measures | Geographic level | Years used | Role |
|---|---|---|---|---|
| **CDC PLACES** | Poor mental health (MHLTH), depression (DEPRESSION) — model-based prevalence | County | 2018–2023 (by data year) | **Need** |
| **U.S. Census ACS (5-yr)** | Population, median income, poverty, unemployment, race/ethnicity | County | rolling 5-yr windows | **Controls** |
| **SAMHSA N-MHSS / N-SUMHSS** | Facility type (wards), inpatient psychiatric beds, client counts (caseload) | Facility → county | available survey years | **Capacity** |
| **HRSA AHRF** | Registered psychiatrists, hospital counts | County (multi-year file) | 2018–2023 | **Capacity** |

*Comparability caveats for each source are documented in Part 6 (Limitations) — they are central to interpreting the results, not an afterthought.*

---

## Part 1 — Setup

In [ ]:
# Run once if these are not already installed:
# !pip install pandas numpy geopandas matplotlib seaborn requests \
#     shapely pyproj statsmodels scipy pyreadr libpysal esda

In [ ]:
import os
import time
import requests

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import geopandas as gpd
from shapely.geometry import Point

import statsmodels.formula.api as smf
from scipy import stats

import pyreadr  # for reading SAMHSA .rdata public-use files

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

In [ ]:
# ---- Configuration --------------------------------------------------------
# Design decisions live here so the whole analysis can be re-pointed in one place.

# Need-side source:
#   "PLACES" -> CDC PLACES, single consistent methodology, county data 2018-2023
#   "CHR"    -> County Health Rankings, reaches back to ~2011-12 BUT has a
#              methodology break in 2016 (must be disclosed in Part 6)
NEED_SOURCE = "PLACES"

# Primary need measure:
#   "MHLTH"      -> % adults reporting frequent poor mental health (longer series)
#   "DEPRESSION" -> % adults with diagnosed depression (PLACES: 2019 onward)
PRIMARY_NEED = "MHLTH"

# Analysis window, expressed as underlying DATA years (not PLACES release labels).
DATA_YEARS = [2018, 2019, 2020, 2021, 2022, 2023]

# Where raw downloaded files live (PLACES CSVs, SAMHSA .rdata, AHRF, shapefiles).
DATA_DIR = "data"

# Census API key — DO NOT hard-code a key in a submitted notebook.
# Get one free at https://api.census.gov/data/key_signup.html and set it via:
#   export CENSUS_API_KEY="your_key"   (shell)   or in your environment.
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")

os.makedirs(DATA_DIR, exist_ok=True)
print("Need source:", NEED_SOURCE, "| primary need:", PRIMARY_NEED)
print("Data years :", DATA_YEARS)

In [ ]:
# ---- Helpers --------------------------------------------------------------

def fips5(series):
    """Coerce a county identifier to a clean 5-digit FIPS string."""
    return series.astype(str).str.extract(r"(\d+)")[0].str.zfill(5)

def zscore_within(df, value_col, group_col="year"):
    """Standardize a column within each group (e.g., within each year)."""
    g = df.groupby(group_col)[value_col]
    return (df[value_col] - g.transform("mean")) / g.transform("std")

## Part 2 — Data acquisition and cleaning

Each subsection loads one raw source, cleans it, and reduces it to a **tidy county–year frame** keyed on `county_fips` and `year`. These are merged into a single panel in Part 3.

### 2A — CDC PLACES (need)
*To build next.* Load each PLACES county release in `DATA_YEARS`, keep `MHLTH` (and `DEPRESSION`), pivot to one row per county-year. Output: `need_panel[county_fips, year, poor_mental_health_pct, depression_pct]`.

### 2B — Census ACS 5-year (controls)
*To build.* Loop the ACS API over the relevant 5-year windows; compute poverty rate, unemployment rate, median income, and race/ethnicity shares. Output: `acs_panel[county_fips, year, ...controls]`.

### 2C — SAMHSA N-MHSS / N-SUMHSS (capacity: wards, beds, caseload)
*To build.* For each available survey year, read the PUF, classify facilities by type (psychiatric hospital / separate inpatient psych unit / state hospital = "wards"), sum inpatient psychiatric beds and client counts, aggregate to county. Output: `samhsa_panel[county_fips, year, n_psych_facilities, inpatient_psych_beds, clients, ...]`.

> **Next-step dependency:** paste the column list from your SAMHSA file so we map the exact bed / client / facility-type / county variables before writing this cell.

### 2D — HRSA AHRF (capacity: psychiatrists, hospitals)
*To build.* Read the AHRF county file (internally multi-year), extract psychiatrist counts and hospital counts for each year in `DATA_YEARS`. Output: `ahrf_panel[county_fips, year, psychiatrists, hospitals]`.

## Part 3 — Build the county–year panel
*To build.* Standardize `county_fips` across sources, handle the Connecticut county→planning-region change, align years, and merge 2A–2D into one tidy panel. Document any rows lost in the merge.

## Part 4 — Need–capacity mismatch index
*To build.* Standardize each component **within year** (so the index is a relative measure each year, comparable across counties). Build a composite capacity score from beds + psychiatrists + facilities, then:

`mismatch = z(need) − z(capacity)`, computed per county-year. High = high need relative to local treatment reach.

## Part 5 — Analysis and visualization
*To build.* (1) National trend in average mismatch over time; (2) county choropleth maps, early vs late year; (3) counties with the largest increase in mismatch; (4) regression of mismatch on poverty / unemployment / race controls with year and state fixed effects.

## Part 6 — Limitations

These are integral to interpreting the index, not caveats tacked on at the end:

- **PLACES is model-based and not intended for trend or policy evaluation.** Cross-release changes can reflect BRFSS/modeling methodology rather than real change; the *release year* is not the *data year* (e.g., the 2024 release uses 2022 BRFSS).
- **County mental-health prevalence does not exist before ~2017–18**, which sets the floor of the need series. (Reaching ~2011 would require switching to County Health Rankings and absorbing a 2016 methodology break — see the `NEED_SOURCE` toggle.)
- **SAMHSA is a repeated cross-section, not a panel of fixed facilities**; the survey was redesigned in 2021 (N-MHSS → N-SUMHSS) and bed counts are collected only every other year.
- **ACS 5-year windows overlap**, so adjacent years are not independent.
- **Connecticut switched from counties to planning regions** within the window, requiring a crosswalk or exclusion.
- **The mismatch index is relative, not absolute** — it ranks counties within each year rather than measuring an absolute adequacy threshold.

## Part 7 — Conclusion and references
*To build.*

**References** *(to compile)* — CDC PLACES; U.S. Census Bureau ACS; SAMHSA N-MHSS / N-SUMHSS; HRSA Area Health Resources Files; County Health Rankings & Roadmaps.